# API Bitget

In [ ]:
# !pip install redis

In [1]:
import ccxt # pip install ccxt
import redis # pip install redis
import pandas as pd # pip install pandas
import json # pip install json
from datetime import datetime # Thu vien built-in

# Thiết lập thông tin tài khoản và sàn giao dịch
exchange = ccxt.bitget({ }) # Ko can dua API Key, API Secret

# Thiết lập cặp tiền tệ và thời gian
symbol = 'BTC/USDT'
timeframe = '1m'

# Lấy dữ liệu giá
ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe) # Ham dua Symbol va Timeframe

print(type(ohlcv))
ohlcv

<class 'list'>


[[1763146140000, 95275.01, 95315.43, 95274.01, 95296.69, 1.859918],
 [1763146200000, 95296.69, 95547.88, 95296.69, 95436.92, 10.14995],
 [1763146260000, 95436.92, 95467.04, 95348.13, 95442.92, 46.619663],
 [1763146320000, 95442.92, 95442.92, 95289.19, 95351.51, 6.452991],
 [1763146380000, 95351.51, 95474.42, 95336.01, 95446.33, 11.490842],
 [1763146440000, 95446.33, 95456.33, 95324.42, 95324.42, 5.866165],
 [1763146500000, 95324.42, 95343.89, 95238.68, 95317.39, 2.938498],
 [1763146560000, 95317.39, 95317.39, 95163.0, 95167.2, 3.156241],
 [1763146620000, 95167.2, 95243.18, 95167.2, 95179.01, 1.503596],
 [1763146680000, 95179.01, 95240.29, 95154.28, 95203.67, 2.729108],
 [1763146740000, 95203.67, 95203.68, 94964.31, 95076.54, 67.539935],
 [1763146800000, 95076.54, 95181.48, 95042.7, 95113.48, 8.292969],
 [1763146860000, 95113.48, 95120.32, 94917.02, 94984.08, 9.670913],
 [1763146920000, 94984.08, 95109.97, 94946.72, 95031.84, 5.006216],
 [1763146980000, 95031.84, 95031.84, 94932.01, 949

# Chuyen du lieu list sang dataframe

In [ ]:
# Chuyển dữ liệu sang DataFrame
df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
print(df.info())
df

# Chuyen du lieu

In [ ]:
# Chuyển đổi kiểu dữ liệu của giá trị trong DataFrame
df['timestamp'] = df['timestamp'].astype(int)
df['open'] = df['open'].astype(float)
df['high'] = df['high'].astype(float)
df['low'] = df['low'].astype(float)
df['close'] = df['close'].astype(float)
df['volume'] = df['volume'].astype(float)

print(df.info())
df

# Filter

In [ ]:
# Tạo tín hiệu mua dựa trên điều kiện volume lớn hơn 10
df['Buy_Signal'] = df['volume'] > 10
df['Sell_Signal'] = df['volume'] < 10

In [ ]:
df

# Lay record moi nhat de day sang redis

In [ ]:
# Tạo dữ liệu 1 record moi nhat để đẩy qua Redis
data = {
    'symbol': symbol,
    'close': float(df['close'].iloc[-1]),
    'timestamp': int(df['timestamp'].iloc[-1]),
    'InsertDate': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

print(data)

# Day sang bo nho tam Redis

In [ ]:
data

In [ ]:
import redis

# Thiết lập kết nối Redis
redis_client = redis.Redis(host='localhost', port=6379, db=12) # FX, Crypto, Chung khoan

# Tên Hash trong Redis
hash_name = 'Chien luoc Crypto SUI/USDT New 30.05' # OF se doc hash nay

# Ghi từng cặp key-value vào Hash
for key, value in data.items():
    redis_client.hset(hash_name, key, json.dumps(value))

# Doc Redis (Cai nay nam trong OF)

In [ ]:
# Đọc lại dữ liệu từ Redis Hash
retrieved_data = {key.decode(): json.loads(value.decode()) for key, value in redis_client.hgetall(hash_name).items()}
print("Dữ liệu trong Redis Hash:")
print(retrieved_data)

# Lay dict

In [ ]:
hash_name = "OG_ML_CK_MA10, MA20"

In [ ]:
# Đọc lại dữ liệu từ Redis Hash
retrieved_data = {key.decode(): value.decode() for key, value in redis_client.hgetall(hash_name).items()}
print("Dữ liệu trong Redis Hash:")
print(retrieved_data)

# Chien luoc dua vao SMA: Close > SMA10 Va SMA10 > SMA50 => Buy, Close < SMA10 va SMA10 < SMA50 => Sell

In [ ]:
df['SMA2'] = df['close'].rolling(window=2).mean()
df['SMA20'] = df['close'].rolling(window=20).mean()
df['SMA50'] = df['close'].rolling(window=50).mean()

df["Buy_Signal"] = (df['close'] > df['SMA2']) & (df['SMA2'] > df['SMA20'])
df["Sell_Signal"] = (df['close'] < df['SMA2']) & (df['SMA2'] < df['SMA20'])

df

In [ ]:
data = df.iloc[-1].to_dict()
data